In [240]:
# import all libraries
import pandas as pd
import numpy as np
import re
from textblob import TextBlob
from nltk.stem import WordNetLemmatizer, PorterStemmer, LancasterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from autocorrect import Speller

In [241]:
# import and read dataset
df = pd.read_csv('../datasets/Fin_lab-PRProject_dataset.csv')
df.head()

,Unnamed: 0,recommendationid,language,review,Reaction
0,0,77057085,english,Is good. Do play.,0
1,1,77052689,english,AAAAAAAA,0
2,2,77049252,english,Fun game,1
3,3,77049089,english,"Great game, worth every penny!",0
4,4,35101272,english,Like,0


In [242]:
# get only 1000 random row samples
sample_df = df.sample(n=1000, random_state=42)
sample_df.head()

,Unnamed: 0,recommendationid,language,review,Reaction
30583,83,27076016,english,This game has got to be one of the most memora...,0
28537,37,28045907,english,The weirdest shit I've ever played and yet ins...,1
11192,92,21598005,english,This game + the original Binding of Isaac got ...,1
18237,37,16024475,english,Binding of Isaac: Cumtopia (REMASTERED),0
16235,35,52680381,english,great,1


In [243]:
# check the dataset null/nan values
sample_df[sample_df['review'].isnull()]

,Unnamed: 0,recommendationid,language,review,Reaction


In [244]:
# check types in 'review' column
sample_df['review'].apply(type).value_counts()

review
<class 'str'>    1000
Name: count, dtype: int64

In [245]:
# contraction mapping
CONTRACTION_MAP = {
    "ain't": "is not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "i've" : "i have",
    "i'd" : "i had",
    "you've" : "you have",
    "you'd" : "you had",
    "we've" : "we have",
    "we'd" : "we had",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "wasn't" : "was not",
    "that's": "that is",
    "you'll" : "you will"
}

In [246]:
# MODULE DECLARATIONS
spellChecker = Speller(lang='en')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# HELPER FUNCTIONS

# tokenize text
def tokenize_text(text):
    word_tokens = word_tokenize(text)
    return word_tokens

# clear rows that has null values
def clear_null_values(text):
    if not isinstance(text, str):
        return ""
    return text

# remove punctuations
def remove_punctuations(text):
    punctuations = '''!()-[]{};:'.'"\,<>./?@#$%^–—&_0123456789~*+'''
    # remove punctuations from the text
    no_punct = ""
    for char in text:
        if char not in punctuations:
            no_punct = no_punct + char
    return no_punct

# remove repeating characters
def remove_repeating_characters(tokens):
    repeatedPattern = re.compile(r'(\w*)(\w)\2(\w*)')
    matchSubstitution = r'\1\2\3'
    def replace(oldWord):
        if wordnet.synsets(oldWord):
            return oldWord
        newWord = repeatedPattern.sub(matchSubstitution, oldWord)
        return replace(newWord) if newWord != oldWord else newWord
    correctTokens = [replace(word) for word in tokens]
    return correctTokens

# remove contractions
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP):

    contractions_pattern = re.compile('({})'.format('|'.join(contraction_mapping.keys())), flags=re.IGNORECASE|re.DOTALL)

    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        expanded_contraction = contraction_mapping.get(match) if contraction_mapping.get(match) else contraction_mapping.get(match.lower())
        expanded_contraction = first_char + expanded_contraction[1:]
        return expanded_contraction
    
    expanded_text = contractions_pattern.sub(expand_match, text)
    expanded_text = re.sub("'", "", expanded_text)
    return expanded_text

# remove stopwords
myWordDictionary = []
def remove_stopwords_orig(text):
    data = text
    stopWords = set(stopwords.words('english'))
    words = data

    wordsFiltered = []
    for w in words:
        if w not in stopWords:
            myWordDictionary.append(w)
            wordsFiltered.append(w)
            wordsFiltered.append(' ')

    return "".join(wordsFiltered)



<>:21: SyntaxWarning: invalid escape sequence '\,'
<>:21: SyntaxWarning: invalid escape sequence '\,'
C:\Users\mosqu\AppData\Local\Temp\ipykernel_13640\1254602377.py:21: SyntaxWarning: invalid escape sequence '\,'
  punctuations = '''!()-[]{};:'.'"\,<>./?@#$%^–—&_0123456789~*+'''


In [247]:
# CLEAN TEXT PIPELINE
def clean_text_pipeline(text, do_spellcheck=True):
    # 1 - handle nulls
    text = clear_null_values(text)

    # 2 - expand contractions
    text = expand_contractions(text)

    # 3 - remove punctuations
    text = remove_punctuations(text)

    # 4 - turn all text to lowercase
    text = text.lower()

    # 5 - tokenize text
    tokens = tokenize_text(text)

    # 6 - remove repeating characters
    tokens = remove_repeating_characters(tokens)

    # 7 - spell check
    if do_spellcheck:
        tokens = [spellChecker(word) for word in tokens]

    # 8 - remove stopwords
    tokens = remove_stopwords_orig(tokens)

    # 9 - stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # 10 - lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # 11 - join back the cleaned text into a single string
    final_cleaned_text = ''.join(tokens)

    return final_cleaned_text.strip()


In [248]:
sample_df['cleaned'] = sample_df['review'].apply(lambda x: clean_text_pipeline(x, do_spellcheck=True))

In [249]:
sample_df.head()

,Unnamed: 0,recommendationid,language,review,Reaction,cleaned
30583,83,27076016,english,This game has got to be one of the most memora...,0,game got one memorable games played truly with...
28537,37,28045907,english,The weirdest shit I've ever played and yet ins...,1,widest shit ever played yet instantly fun defi...
11192,92,21598005,english,This game + the original Binding of Isaac got ...,1,game original binding isaac got rougelike game...
18237,37,16024475,english,Binding of Isaac: Cumtopia (REMASTERED),0,binding isaac utopia remastered
16235,35,52680381,english,great,1,great


In [250]:
# BAG OF WORDS EXTRACTION

# corpus is the collection of text docs
# n of n-gram can be modified
# max feature is limited to 500 to keep top 500 most frequent words, can be changed though
def bow_extractor(corpus, ngram_range=(1,1), max_features=500):
    vectorizer = CountVectorizer(min_df=1, ngram_range=ngram_range, max_features=max_features)
    features = vectorizer.fit_transform(corpus)
    return vectorizer, features

# display features
def display_features(features, feature_names):
    df = pd.DataFrame(data=features, columns=feature_names)
    print(df)

# save features to csv
def save_features(features, feature_names, filename):
    df = pd.DataFrame(data=features, columns=feature_names)
    df.to_csv(filename, index=False)
    print(f"✅ Features extracted and saved to {filename}")

# TFIDF EXTRACTION
def tfidf_transformer(bow_matrix):
    t = TfidfTransformer(norm='l2', smooth_idf=True, use_idf=True)
    tfidf_matrix = t.fit_transform(bow_matrix)
    return t, tfidf_matrix

In [251]:
# Extract BoW features
corpus = sample_df['cleaned'].tolist()
bow_vectorizer, bow_features = bow_extractor(corpus)


In [252]:
# get feature names
feature_names = bow_vectorizer.get_feature_names_out()

display_features(bow_features.todense(), feature_names)

     absolutely  achievements  actually  addicted  addiction  addictive  \
0             0             0         0         0          0          0   
1             0             0         0         0          0          0   
2             0             0         0         0          0          0   
3             0             0         0         0          0          0   
4             0             0         0         0          0          0   
..          ...           ...       ...       ...        ...        ...   
995           0             0         0         0          0          0   
996           0             0         0         0          0          0   
997           0             0         0         0          0          0   
998           0             0         0         0          0          0   
999           0             0         0         0          0          0   

     adding  adds  aesthetic  afterbirth  ...  worth  would  wrong  xbox  \
0         0     0      

In [253]:
# save BOW features to csv
save_features(bow_features.toarray(), feature_names, '../text-mining/mosqueda_bow_features.csv')

✅ Features extracted and saved to ../text-mining/mosqueda_bow_features.csv


In [254]:
# TFIDF extraction implementation using BOW features
tfidf_transform, tfidf_features = tfidf_transformer(bow_features)
features_idf = np.round(tfidf_features.todense(), 2)
display_features(features_idf, feature_names)

     absolutely  achievements  actually  addicted  addiction  addictive  \
0           0.0           0.0       0.0       0.0        0.0        0.0   
1           0.0           0.0       0.0       0.0        0.0        0.0   
2           0.0           0.0       0.0       0.0        0.0        0.0   
3           0.0           0.0       0.0       0.0        0.0        0.0   
4           0.0           0.0       0.0       0.0        0.0        0.0   
..          ...           ...       ...       ...        ...        ...   
995         0.0           0.0       0.0       0.0        0.0        0.0   
996         0.0           0.0       0.0       0.0        0.0        0.0   
997         0.0           0.0       0.0       0.0        0.0        0.0   
998         0.0           0.0       0.0       0.0        0.0        0.0   
999         0.0           0.0       0.0       0.0        0.0        0.0   

     adding  adds  aesthetic  afterbirth  ...  worth  would  wrong  xbox  \
0      0.00  0.00      

In [255]:
# save TFIDF features to csv
save_features(features_idf, feature_names, '../text-mining/mosqueda_tfidf_features.csv')

✅ Features extracted and saved to ../text-mining/mosqueda_tfidf_features.csv
